---
title: "Parsing VASP and Gaussian's Output Files with Python"
date: "2026-09-13"
---

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1V8eIcKkPiZPYpxBgEKTE9QiSL3tW3IRT?usp=sharing)

# This is example code for parsing VASP and Gaussian file formats using the Pymatgen Python library


## Generic text Parsing Gaussian file for Mulliken Energy

In [ ]:
# import regular expression module
import re

# Read the file
with open("gaussian/tddft.log") as f:
    lines = f.readlines()

# We will parse the Mulliken Charges for all atoms
# Find the pattern
start_pattern = re.compile(r'Mulliken charges:')

# The pattern at the end
end_pattern = re.compile(r'Sum of Mulliken charges')


We will use `re.match()` to check whether the line contains our start pattern.

In [ ]:
# Find line with start pattern of Mulliken charges
for idx, line in enumerate(lines): # loops over the lines
    if re.match(start_pattern, line.strip()):
        break
print(idx)

To stop parsing we will again use `re.match()` to match the end pattern. We will apply lines to the list.

In [ ]:
# start parse lines
idx = idx + 2
line = lines[idx]

parsed_lines = []
while not(re.match(end_pattern, line.strip())):
    parsed_lines.append(line.strip())
    idx += 1
    line = lines[idx]

In [ ]:
# Clean up the data
data =[]
for line in parsed_lines:
    clean_line = line.strip().split()
    data.append({"number": int(clean_line[0]), 
     "atom": clean_line[1], 
     "charge": float(clean_line[2])})

In [ ]:
# Create a table with pandas
import pandas as pd

pd.DataFrame(data)

# save as CSV file
df = pd.DataFrame(data)
df.to_csv("parsing_MC.csv", index = False)

# Parsing VASP and Gaussian Output files with pymatgen
We can also parse any text file created by VASP with pymatgen. Below is an example to parse the `vasprun.xml` output file.

## Parsing band Structure data from vasprun.xml
Here we will look at the available methods for Vasprun from [pymatgen.io.vasp package](https://pymatgen.org/pymatgen.io.vasp.html).

In [ ]:
# Load vasprun from pymatgen
from pymatgen.io.vasp import Vasprun, BSVasprun

# read the xml file
vasprun = Vasprun("vasp/vasprun.xml/vasprun.xml")

# Plot Band structure
from pymatgen.io.vasp import Vasprun, BSVasprun
from pymatgen.electronic_structure.plotter import BSPlotter

v = BSVasprun("vasp/vasprun.xml/vasprun.xml")
bs = v.get_band_structure(kpoints_filename="vasp/KPOINTS",line_mode=True)
plt = BSPlotter(bs)
plt.get_plot(vbm_cbm_marker=True,ylim=(-3,3))
plt.show()

## Plotting the density of states

The electronic density of states (DOS) describes how many electronic states are available at a given energy. It is a useful tool for analyzing the electronic structure of materials, identifying band gaps, and distinguishing between metallic and insulating behavior. The DOS can also reveal the contribution of different atoms and orbitals to the electronic states (https://vasp.at/wiki/Category:Density_of_states).


Pymatgen also had module to plot. Here we will use the DoSPlotter, [pymatgen.electronic_structure package](https://pymatgen.org/pymatgen.electronic_structure.html), to plot density of states. 



In [ ]:
# plotting density of states
from pymatgen.electronic_structure.plotter import DosPlotter

tdos = vasprun.tdos
plotter = DosPlotter(sigma=0.1)
plotter.add_dos("Total DOS", tdos)
plotter.show()

## Parsing molecule structure from gaussian.log
The modules to parse gaussian are available in pymatgen. Other codes which can be parsed are listed here (https://pymatgen.org/pymatgen.html).

In [ ]:
# We import the module load parse gaussian output files
from pymatgen.io.gaussian import GaussianOutput

# Reading the log file
gout = GaussianOutput("gaussian/tddft.log")

# Final structure
gout.final_structure

# Save structure data as CIF file
from pymatgen.core import Structure
structure = gout.final_structure

structure.to(filename="molecule_summary.xyz")